# ema-first-moment — ex3: dual m and v EMA update from one gradient list

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `ema-first-moment`. Running the final beacon cell reports progress against the `Optimizer: Adam EMA first moment` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA first moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-first-moment`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-first-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA first moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Dual m + v update in one pass — Adam's full state buffers

Ex1 updated `m` (first moment, EMA of gradient). Ex2 updated `v` (second moment, EMA of squared gradient). The deepening move runs BOTH updates in a single helper, mutating both buffers from the same gradient list — the exact pattern `torch.optim.Adam.step` uses inside its per-parameter loop.

```python
for m, v, g in zip(m_list, v_list, grad_list):
    m.copy_(beta1 * m + (1 - beta1) * g)         # first moment
    v.copy_(beta2 * v + (1 - beta2) * g.pow(2))  # second moment
```

**Order doesn't matter — but use one pass.** Since `m` and `v` are independent buffers (no cross-dependency), you can update them in either order. Combining them in ONE loop over parameters is the memory-locality win: each param's `(m, v, g)` triple is in cache together. Real Adam implementations always co-locate the update.

**Why `g.pow(2)` not `g * g`.** Both work numerically. `pow(2)` is fused in PyTorch's CUDA kernel and skips one tensor allocation vs `g * g` (which materializes a temporary). For tight inner loops, `pow(2)` is the canonical choice.

**No bias correction here.** The dual-update helper just runs the EMAs. Bias correction (`m_hat`, `v_hat`) is the NEXT step in Adam and lives in a separate atom — keeping concerns separate is the whole point of the per-atom skeleton.

### Exercise 3 — dual m and v EMA update from one gradient list

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply both Adam EMA recurrences in a single co-located loop — `m = beta1*m + (1-beta1)*g` and `v = beta2*v + (1-beta2)*g**2` — so each parameter's `(m, v, g)` triple is touched once.
> Keywords: adam, first-moment, second-moment, co-located-update
> ```

**KCs targeted:** `co-locate-m-and-v-update-in-one-loop`, `second-moment-uses-g-squared-not-g`

Implement `ex3_update_m_and_v(m_list, v_list, grad_list, beta1, beta2)`. Apply Adam's first-moment AND second-moment EMA updates in one combined pass.

Inputs:
- `m_list`: list of first-moment buffers (one tensor per param).
- `v_list`: list of second-moment buffers (one tensor per param).
- `grad_list`: list of gradient tensors (one per param).
- `beta1`: float — EMA decay for `m`.
- `beta2`: float — EMA decay for `v`.

All three lists have the same length, and the i-th entries share a shape.

For each triple `(m, v, g)`:

1. `m.copy_(beta1 * m + (1 - beta1) * g)` — first moment.
2. `v.copy_(beta2 * v + (1 - beta2) * g.pow(2))` — second moment.

Mutate `m_list` and `v_list` IN PLACE. Return `None`.

Both updates MUST use `.copy_()` so the original buffer tensor identity (its `data_ptr()`) is preserved across the call.

In [ ]:
def ex3_update_m_and_v(m_list, v_list, grad_list, beta1, beta2):
    for m, v, g in zip(m_list, v_list, grad_list):
        m.copy_(beta1 * m + (1.0 - beta1) * g)
        v.copy_(beta2 * v + (1.0 - beta2) * g.pow(2))


<details><summary>Solution</summary>

```python
def ex3_update_m_and_v(m_list, v_list, grad_list, beta1, beta2):
    for m, v, g in zip(m_list, v_list, grad_list):
        m.copy_(beta1 * m + (1.0 - beta1) * g)
        v.copy_(beta2 * v + (1.0 - beta2) * g.pow(2))
```

**Both updates are independent.** No cross-dependency between m and v — each only reads from itself and `g`. Running them in one loop (over params) is just memory-locality / cache friendliness; the math is identical to two separate passes.

**`.copy_()` not assignment.** Plain `m = ...` rebinds the local name in the function scope; the caller's `m_list[i]` still points to the original tensor. `m.copy_(...)` mutates the existing storage, so the caller sees the update.

**`g.pow(2)` vs `g * g`.** Same result; `pow(2)` is one CUDA kernel and one allocation. `g * g` materializes a temporary. In Adam's per-step inner loop this matters for very large models.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()